This notebook trains and evaluates a YOLOv8 object detector for aerial traffic images using `Data_2_v2`.

The workflow compares:

1. A pretrained COCO YOLOv8n model, evaluated only on COCO traffic classes remapped to this project: `car`, `bus`, `truck`.
2. The same YOLOv8n model fine-tuned on the annotated aerial intersection dataset.

Run this notebook on the GPU PC, not on the Mac.

In [ ]:
# Install once on the training PC if needed.
# !pip install ultralytics pandas matplotlib seaborn pillow opencv-python

from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Exercise_2":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_DIR = PROJECT_ROOT / "Data_2_v2"
EXERCISE_DIR = PROJECT_ROOT / "Exercise_2"
SCRIPT = EXERCISE_DIR / "exercise2_yolo_pipeline.py"

# Use 0 for the first NVIDIA GPU. Use 'cpu' only for quick checks, not training.
DEVICE = "0"

print("Project:", PROJECT_ROOT)
print("Dataset:", DATASET_DIR)
print("Script:", SCRIPT)


The dataset was prepared in Exercise 1. It contains two source videos, YOLO labels, and train/validation/test splits.

In [ ]:
report = json.loads((DATASET_DIR / "dataset_report.json").read_text())
print("Total images:", report["total_images"])
print("Total boxes:", report["total_boxes"])
print(json.dumps(report["subset_stats"], indent=2))

This evaluates the pretrained YOLOv8n COCO model on the test set. COCO class IDs are remapped before metric calculation: `car=2 -> 0`, `bus=5 -> 1`, `truck=7 -> 2`.

In [ ]:
subprocess.run([
    sys.executable, str(SCRIPT),
    "--baseline",
    "--imgsz", "960",
    "--conf", "0.25",
    "--device", DEVICE,
], check=True)

This is the slow step. The default settings are a good balance for the dataset size. Early stopping is enabled with `--patience 15`, so training stops if validation performance does not improve for 15 epochs. If the GPU runs out of memory, reduce `--batch` to `8` or `--imgsz` to `640`.

In [ ]:
subprocess.run([
    sys.executable, str(SCRIPT),
    "--train",
    "--epochs", "80",
    "--imgsz", "960",
    "--batch", "16",
    "--patience", "15",
    "--device", DEVICE,
], check=True)

This evaluates `best.pt` on the held-out test set using the same custom metric code as the pretrained baseline, then creates a CSV, comparison plot, and example detections for the report.

In [ ]:
subprocess.run([
    sys.executable, str(SCRIPT),
    "--evaluate-finetuned",
    "--imgsz", "960",
    "--conf", "0.25",
    "--device", DEVICE,
], check=True)

subprocess.run([sys.executable, str(SCRIPT), "--compare"], check=True)

subprocess.run([
    sys.executable, str(SCRIPT),
    "--examples",
    "--imgsz", "960",
    "--conf", "0.25",
    "--device", DEVICE,
], check=True)